# 9.4 Sigma Sweep 2 (seeds = [0, 1, 2])

Runs the same sweep as `simulated_data_pipeline_aug1.py`'s `__main__` block (4 models × 7 shift_sigmas × seeds, n_genes=200, using `train_and_compute_rho_r2`), but with `seeds = [0, 1, 2]` instead of the `[3, 4]` currently set in the script. This lets the partner's seed range be produced independently while the script itself stays on `[3, 4]`.

Writes to the same locations the script would (`shift_sigma_sweep/<MMDD>/trained/` and `shift_sigma_sweep/<MMDD>/sweep_log_<sigmas>_<seeds>.csv`), so results merge cleanly with a partner's run of the same script.

## Import pipeline module

In [ ]:
import sys
sys.path.insert(0, "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Data Generation")

import os
import numpy as np
import pandas as pd
from datetime import datetime

from simulated_data_pipeline_aug1 import train_and_compute_rho_r2

## Run sweep

In [ ]:
# ---- sweep ----
# shift distribution is MVN(0, shift_sigma^2 * I) in n_z_dims
shift_sigmas = [0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]
seeds = [0, 1, 2]
decipher_seeds = [1]  # one decipher_seed
n_z_dims = 3
n_samples = 500
n_genes = 200
biological_sigma = 0.1
models = {
    "Decipher-VZ": "decipher_vz",
    "Decipher-VZ2": "decipher_vz2",
    "Decipher-MF": "decipher_mf",
    "Base Decipher": "decipher",
}
colors = {
    "Decipher-VZ": "#2E7D32",
    "Decipher-VZ2": "#1565C0",
    "Decipher-MF": "#F9A825",
    "Base Decipher": "#C62828",
}

pipeline_dir = "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Data Generation"
today = datetime.now().strftime("%m%d")
log_dir = os.path.join(pipeline_dir, "..", "Simulated Adata", "shift_sigma_sweep", today)
os.makedirs(log_dir, exist_ok=True)
csv_path = os.path.join(log_dir, f"sweep_log_{shift_sigmas}_{seeds}.csv")

run_log = []
for name, model in models.items():
    for shift_sigma in shift_sigmas:
        for sd in seeds:
            for decipher_seed in decipher_seeds:
                record = {
                    "model": name, "n_z_dims": n_z_dims,
                    "shift_sigma": shift_sigma, "seed": sd,
                    "decipher_seed": decipher_seed,
                }
                try:
                    rho, trained_path, r2_overall, r2_per_gene_median, _ = train_and_compute_rho_r2(
                        model, decipher_seed,
                        shift_sigma=shift_sigma,
                        n_samples=n_samples,
                        n_genes=n_genes,
                        biological_sigma=biological_sigma,
                        seed=sd,
                        n_z_dims=n_z_dims,
                    )
                    record.update({
                        "rho": rho, "trained_h5ad": trained_path,
                        "r2_overall": r2_overall,
                        "r2_per_gene_median": r2_per_gene_median,
                        "error": None,
                    })
                except Exception as e:
                    print(f"[{name}] n_z_dims={n_z_dims} shift_sigma={shift_sigma} "
                          f"seed={sd} decipher_seed={decipher_seed} "
                          f"failed: {type(e).__name__}: {e}")
                    record.update({
                        "rho": np.nan, "trained_h5ad": None,
                        "r2_overall": np.nan, "r2_per_gene_median": np.nan,
                        "error": f"{type(e).__name__}: {e}",
                    })
                run_log.append(record)
                pd.DataFrame(run_log).to_csv(csv_path, index=False)

run_log_df = pd.DataFrame(run_log)
run_log_df

## Correlation Plot

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

log_dir = "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Adata/shift_sigma_sweep/0804"
trained_dir = os.path.join(log_dir, "trained")
models = {
    "Decipher-VZ": "decipher_vz",
    "Decipher-VZ2": "decipher_vz2",
    "Decipher-MF": "decipher_mf",
    "Base Decipher": "decipher",
}
colors = {
    "Decipher-VZ": "#2E7D32",
    "Decipher-VZ2": "#1565C0",
    "Decipher-MF": "#F9A825",
    "Base Decipher": "#C62828",
}

# Pool the partner's seeds=[0,1,2] sweep (committed to git) with this machine's
# fresh seeds=[3,4] sweep. Each CSV's `trained_h5ad` column stores an absolute
# path from whichever machine ran it (e.g. /Users/jihyunpark/... for the
# seeds=[0,1,2] rows), which doesn't resolve here -- but the .h5ad files
# themselves are present locally (shift_sigma_sweep/0804/trained/ is
# git-whitelisted), so we reconstruct the local path from the known naming
# convention instead of trusting the stored path.
df_012 = pd.read_csv(os.path.join(log_dir, "sweep_log_[0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]_[0, 1, 2].csv"))
df_34 = pd.read_csv(os.path.join(log_dir, "sweep_log_[0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]_[3, 4].csv"))
run_log_df = pd.concat([df_012, df_34], ignore_index=True)
run_log_df = run_log_df[run_log_df["error"].isna()].copy()
run_log_df["trained_h5ad"] = run_log_df.apply(
    lambda r: os.path.join(trained_dir, f"sigma{r['shift_sigma']}_seed{r['seed']}_{models[r['model']]}.h5ad"),
    axis=1,
)

sigmas_sorted = sorted(run_log_df["shift_sigma"].unique())
x_pos = np.arange(len(sigmas_sorted))  # evenly spaced positions, one per sigma tested

fig, ax = plt.subplots(figsize=(7, 5))
for name in models:
    sub = run_log_df[run_log_df["model"] == name]
    stats = sub.groupby("shift_sigma")["rho"].agg(["mean", "std"]).reindex(sigmas_sorted)
    mean = stats["mean"].to_numpy()
    sd_ = stats["std"].to_numpy()
    ax.plot(x_pos, mean, marker="o", color=colors[name], label=name, linewidth=2)
    ax.fill_between(x_pos, mean - sd_, mean + sd_, color=colors[name], alpha=0.18)

ax.set_xticks(x_pos)
ax.set_xticklabels([str(s) for s in sigmas_sorted])
ax.set_xlabel(r"batch-shift noise  $\sigma$   (shifts $\sim \mathcal{N}(0,\sigma^2)$, normalized by $\sqrt{3}$)")
ax.set_ylabel(r"Spearman $|\rho|$  (decipher_time vs latent_t)")
ax.set_title("Trajectory recovery vs batch noise (seeds = [0, 1, 2, 3, 4], n_genes=200)")
ax.set_ylim(0, 1); ax.axhline(0, color="0.8", lw=0.8)
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("sigma_vs_correlation_9.4.png", dpi=200)
plt.show()


In [ ]:
rho_summary = run_log_df.groupby("shift_sigma")["rho"].agg(["mean", "std"]).reindex(sigmas_sorted).round(3)
rho_summary


## V-space integration metrics (scib_metrics)

The mentor's check: Z-space should separate batch (silhouette high), V-space should *not* (batches should mix). iLISI, graph connectivity, and (rescaled) batch silhouette are all oriented so **higher = better integration** here.

In [ ]:
import scanpy as sc

def compute_integration_metrics(adata,
                                v_key="decipher_v", batch_key="batch", t_key="latent_t",
                                n_neighbors=30, n_bins=5):
    from scib_metrics.nearest_neighbors import pynndescent
    from scib_metrics import ilisi_knn, graph_connectivity, silhouette_batch
    import pandas as pd
    import numpy as np
    adata.obs["latent_t_bin"] = pd.qcut(
        adata.obs[t_key], q=n_bins, labels=False
    ).astype("category")
    neigh = pynndescent(adata.obsm[v_key], n_neighbors=n_neighbors)
    batches = adata.obs[batch_key].cat.codes.to_numpy()
    bins    = adata.obs["latent_t_bin"].cat.codes.to_numpy()
    return {
        "ilisi": ilisi_knn(neigh, batches, scale=True),
        "graph_conn": graph_connectivity(neigh, labels=bins),
        "sil_batch": silhouette_batch(adata.obsm[v_key], labels=bins, batch=batches, rescale=True),
    }


In [ ]:
# Only rows that trained successfully have a trained_h5ad to read.
ok_df = run_log_df[run_log_df["error"].isna()].copy()

v_records = []
for _, row in ok_df.iterrows():
    adata = sc.read_h5ad(row["trained_h5ad"])
    m = compute_integration_metrics(adata, v_key="decipher_v", batch_key="batch", t_key="latent_t",
                                     n_neighbors=30, n_bins=5)
    m.update({"model": row["model"], "shift_sigma": row["shift_sigma"], "seed": row["seed"]})
    v_records.append(m)

v_results_df = pd.DataFrame(v_records)
v_results_df.to_csv(os.path.join(log_dir, "v_space_integration_metrics_9.4.csv"), index=False)
v_results_df


### V-space metrics vs sigma

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 4))
panels = [
    ("rho", "Spearman |rho|", run_log_df),
    ("ilisi", "iLISI (higher = better mixed)", v_results_df),
    ("graph_conn", "Graph connectivity (higher = better)", v_results_df),
    ("sil_batch", "Inverse silhouette, batch (higher = better)", v_results_df),
]
for ax, (col, title, df) in zip(axes, panels):
    for name in models:
        sub = df[df["model"] == name]
        stats = sub.groupby("shift_sigma")[col].agg(["mean", "std"]).reindex(sigmas_sorted)
        mean = stats["mean"].to_numpy()
        sd_ = stats["std"].to_numpy()
        ax.plot(x_pos, mean, marker="o", color=colors[name], label=name, linewidth=2)
        ax.fill_between(x_pos, mean - sd_, mean + sd_, color=colors[name], alpha=0.15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(s) for s in sigmas_sorted])
    ax.set_xlabel(r"batch-shift noise $\sigma$")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("score")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Model comparison vs batch noise (seeds = [0, 1, 2], n_genes=200)")
fig.tight_layout()
fig.savefig("v_space_metrics_vs_sigma_9.4.png", dpi=200)
plt.show()


In [ ]:
v_summary = (v_results_df
    .groupby(["model", "shift_sigma"])[["ilisi", "graph_conn", "sil_batch"]]
    .agg(["mean", "std"]).round(3))
v_summary


## Per-sigma, per-model detail: Z-space, V-space, X-space

Pools all 5 seeds `[0, 1, 2, 3, 4]` the same way as the Correlation Plot / V-space integration metrics sections above. Each section below picks one representative seed (`seed_for_detail`) per (model, sigma) for the illustrative plots, matching 9.3's convention -- plotting all 5 seeds here would multiply every figure by 5.

In [ ]:
import os
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

log_dir = "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Adata/shift_sigma_sweep/0804"
trained_dir = os.path.join(log_dir, "trained")
models = {
    "Decipher-VZ": "decipher_vz",
    "Decipher-VZ2": "decipher_vz2",
    "Decipher-MF": "decipher_mf",
    "Base Decipher": "decipher",
}
colors = {
    "Decipher-VZ": "#2E7D32",
    "Decipher-VZ2": "#1565C0",
    "Decipher-MF": "#F9A825",
    "Base Decipher": "#C62828",
}

# Pool the partner's seeds=[0,1,2] sweep (committed to git) with this machine's
# fresh seeds=[3,4] sweep. Each CSV's `trained_h5ad` column stores an absolute
# path from whichever machine ran it (e.g. /Users/jihyunpark/... for the
# seeds=[0,1,2] rows), which doesn't resolve here -- but the .h5ad files
# themselves are present locally (shift_sigma_sweep/0804/trained/ is
# git-whitelisted), so we reconstruct the local path from the known naming
# convention instead of trusting the stored path.
df_012 = pd.read_csv(os.path.join(log_dir, "sweep_log_[0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]_[0, 1, 2].csv"))
df_34 = pd.read_csv(os.path.join(log_dir, "sweep_log_[0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]_[3, 4].csv"))
run_log_df = pd.concat([df_012, df_34], ignore_index=True)
run_log_df = run_log_df[run_log_df["error"].isna()].copy()
run_log_df["trained_h5ad"] = run_log_df.apply(
    lambda r: os.path.join(trained_dir, f"sigma{r['shift_sigma']}_seed{r['seed']}_{models[r['model']]}.h5ad"),
    axis=1,
)

def load_run(df, shift_sigma, model_name, seed=None):
    sub = df[(df["shift_sigma"] == shift_sigma) & (df["model"] == model_name)]
    if seed is not None:
        sub = sub[sub["seed"] == seed]
    row = sub.iloc[0]
    return row, sc.read_h5ad(row["trained_h5ad"])

latent_z_cols = ["latent_z0", "latent_z1", "latent_z2"]
model_names = list(models.keys())
seed_for_detail = 0


## Z-space: ground truth vs. learned decipher_z

In [ ]:
def plot_z_space(shift_sigma, seed=seed_for_detail):
    for model_name in model_names:
        row, adata = load_run(run_log_df, shift_sigma, model_name, seed=seed)
        batch_labels = adata.obs["batch"].astype(str)

        sil_ground_truth = silhouette_score(
            adata.obs[latent_z_cols].values, batch_labels,
            sample_size=min(5000, adata.n_obs), random_state=42,
        )
        sil_learned = silhouette_score(
            adata.obsm["decipher_z"], batch_labels,
            sample_size=min(5000, adata.n_obs), random_state=42,
        )
        print(f"{model_name}  sigma={shift_sigma}  seed={seed}  ground truth z silhouette (by batch) = {sil_ground_truth:.4f}")
        print(f"{model_name}  sigma={shift_sigma}  seed={seed}  learned decipher_z silhouette (by batch) = {sil_learned:.4f}")

        adata.obsm["latent_z"] = adata.obs[latent_z_cols].values
        sc.pp.neighbors(adata, use_rep="latent_z", key_added="latent_z_neighbors", random_state=42)
        sc.tl.umap(adata, neighbors_key="latent_z_neighbors", random_state=42)
        adata.obsm["latent_z_umap"] = adata.obsm["X_umap"]

        sc.pp.neighbors(adata, use_rep="decipher_z", random_state=42)
        sc.tl.umap(adata, random_state=42)
        adata.obsm["decipher_z_umap"] = adata.obsm["X_umap"]

        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
        sc.pl.embedding(adata, basis="latent_z_umap", color="batch",
                        ax=axes[0], show=False, title=f"Ground truth z (3D), UMAP ({model_name}, sigma={shift_sigma})")
        sc.pl.embedding(adata, basis="decipher_z_umap", color="batch",
                        ax=axes[1], show=False, title=f"Learned decipher_z, UMAP ({model_name}, sigma={shift_sigma})")
        plt.tight_layout()
        plt.show()


### Sigma = 0.1

In [ ]:
plot_z_space(0.1)


### Sigma = 0.5

In [ ]:
plot_z_space(0.5)


### Sigma = 1.0

In [ ]:
plot_z_space(1.0)


### Sigma = 2.0

In [ ]:
plot_z_space(2.0)


### Sigma = 5.0

In [ ]:
plot_z_space(5.0)


### Sigma = 7.5

In [ ]:
plot_z_space(7.5)


### Sigma = 10.0

In [ ]:
plot_z_space(10.0)


## V-space: colored by batch, latent pseudotime, learned pseudotime

In [ ]:
def plot_v_space(shift_sigma, seed=seed_for_detail):
    for model_name in model_names:
        row, adata = load_run(run_log_df, shift_sigma, model_name, seed=seed)
        sc.pl.embedding(adata, basis="decipher_v", color=["batch", "latent_t", "decipher_time"])
        print(f"{model_name}, sigma={shift_sigma}, seed={row['seed']}, rho = {row['rho']:.4f}")


### Sigma = 0.1

In [ ]:
plot_v_space(0.1)


### Sigma = 0.5

In [ ]:
plot_v_space(0.5)


### Sigma = 1.0

In [ ]:
plot_v_space(1.0)


### Sigma = 2.0

In [ ]:
plot_v_space(2.0)


### Sigma = 5.0

In [ ]:
plot_v_space(5.0)


### Sigma = 7.5

In [ ]:
plot_v_space(7.5)


### Sigma = 10.0

In [ ]:
plot_v_space(10.0)


## X-space: colored by batch

In [ ]:
def plot_x_space(shift_sigma, seed=seed_for_detail, use_umap=False, n_pcs=20):
    for model_name in model_names:
        row, adata = load_run(run_log_df, shift_sigma, model_name, seed=seed)

        sc.pp.pca(adata, n_comps=min(n_pcs, adata.n_vars - 1), random_state=0)
        sc.pl.embedding(adata, basis="pca", color="batch",
                         title=f"{model_name}, sigma={shift_sigma}, X-space PCA")

        if use_umap:
            sc.pp.neighbors(adata, use_rep="X_pca", random_state=42)
            sc.tl.umap(adata, random_state=42)
            sc.pl.embedding(adata, basis="umap", color="batch",
                             title=f"{model_name}, sigma={shift_sigma}, X-space UMAP")


### Sigma = 0.1

In [ ]:
plot_x_space(0.1)


### Sigma = 0.5

In [ ]:
plot_x_space(0.5)


### Sigma = 1.0

In [ ]:
plot_x_space(1.0)


### Sigma = 2.0

In [ ]:
plot_x_space(2.0)


### Sigma = 5.0

In [ ]:
plot_x_space(5.0)


### Sigma = 7.5

In [ ]:
plot_x_space(7.5)


### Sigma = 10.0

In [ ]:
plot_x_space(10.0)
